In [ ]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from src.utils.serialization import load_checkpoint
from src import (
    ConfigLoader, load_feature_config, load_split_config, load_config,
    HydroDataset, create_temporal_split_with_gap,
    Seq2SeqHydro, train_model, predict_autoregressive,
    custom_collate_fn, get_device,
    compute_flow_metrics,
    print_metrics_summary,
    plot_predictions_with_context,
    plot_metrics_by_horizon
)
# Importe também custom_collate_fn e predict_autoregressive onde estiverem

# Configuração
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_PATH = "modelo_hidrologico_v1.pth"

# 1. Carregar Modelo e Metadados
print("📂 Carregando checkpoint...")
model, meta = load_checkpoint(MODEL_PATH, device=DEVICE)

# 2. Carregar seus Novos Dados
# O df_novo DEVE ter as mesmas colunas geradas no treino (médias móveis, etc.)
# df_novo = pd.read_csv(...) 

# 3. Criar o Dataset de Inferência
print("📦 Criando dataset de inferência...")
# Nota: Passamos train_indices=[0] apenas para satisfazer o construtor,
# pois logo em seguida vamos sobrescrever os scalers.
ds_inference = HydroDataset(
    df=df_novo,
    stations=meta["stations"],
    static_attrs=static_attributes_dict, # Precisa ter este dicionário carregado
    train_indices=np.array([0]),         # Dummy index
    forecast_cols=meta["forecast_cols"],
    flow_window_config=meta["flow_window_config"],
    climate_window_config=meta["climate_window_config"],
    temporal_features=meta["temporal_features"],
    api_k_list=[],                       # Não é mais usado
    static_keys=meta["static_keys"],
)

# 4. 💉 INJEÇÃO DE SCALERS (Passo Crucial)
# Substituímos os scalers calculados no df_novo pelos scalers originais do treino.
# Isso garante que a normalização seja consistente.
ds_inference.flow_scalers = meta["flow_scalers"]
ds_inference.climate_scalers = meta["climate_scalers"]
ds_inference.static_scalers = meta["static_scalers"]

# 5. DataLoader
dl_inference = DataLoader(
    ds_inference, 
    batch_size=1, 
    shuffle=False, 
    collate_fn=custom_collate_fn
)

# 6. Executar Previsão
print("🔮 Gerando previsões...")
preds, obs, baseline, g_seq, dates, _ = predict_autoregressive(
    model=model,
    loader=dl_inference,
    decoder_history=meta["decoder_history"],
    decoder_horizon=meta["decoder_horizon"],
    scalers=meta["flow_scalers"], # Importante: passar os scalers do treino para desnormalizar a saída
    stations=meta["stations"],
    clamp_non_negative=True,
    device=DEVICE
)

print("✅ Concluído!")
print(f"Previsões shape: {preds.shape}")